# 05 · Initial conditions vs. continuity

Put 1890 church density and the persistence index in the same regression. If
continuity (not the historical starting level) is what matters, persistence
survives and 1890 density does not. **Manuscript: §3.3 / §4.**

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/cleaned_data_v3.csv")
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "Pop 2010": "pop_2010",
    "Established firms 1989": "firms_1989",
    "Established firms 2022": "firms_2022",
})

for c in ["Index_v2", "church_density_1890", "income_1989_real_2023",
          "pct_hs_1990", "pop_2010", "firms_2022", "firms_1989"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[(df["firms_2022"] > 0) & (df["pop_2010"] > 0) & (df["firms_1989"] > 0)]
df["log_firms_2022"] = np.log1p(df["firms_2022"])
df["log_firms_1989"] = np.log(df["firms_1989"])
df["log_pop_2010"] = np.log(df["pop_2010"])
df[["income_1989_scaled", "pct_hs_1990_scaled"]] = StandardScaler().fit_transform(
    df[["income_1989_real_2023", "pct_hs_1990"]])

d = df.dropna(subset=["Index_v2", "church_density_1890", "income_1989_scaled",
                      "pct_hs_1990_scaled", "log_pop_2010", "log_firms_2022", "State"])
d = d.replace([np.inf, -np.inf], np.nan).dropna()
print(f"N = {len(d)}")

# both measures together: continuity vs. initial level
model = smf.ols(
    "log_firms_2022 ~ church_density_1890 + Index_v2 + income_1989_scaled "
    "+ pct_hs_1990_scaled + log_pop_2010 + C(State)",
    data=d).fit(cov_type="HC3")
print(model.summary())